# Proceso ELT para transferir mensajes de Kafka a la capa Bronze de llamadas

Este proceso ETL tiene como objetivo servir como intermediario para transferir eventos de llamadas almacenados en un clúster de Kafka en la capa Bronze de llamadas. Iniciaremos con el código básico y mejoras posibles se implementan posteriormente. A través de Spark Structured Streaming, se sondearán las particiones del tópico al que el proceso siguiente se suscribe para leer lotes de mensajes escritos, deserializarlos y volcarlos en crudo a la capa ya mencionada. El proceso se divide en los pasos que siguen:

## 0. Creación de SparkSession con conector de Spark para Kafka y tabla Delta (necesario en Jobs de Spark posteriores)

Por supuesto, este paso se omite en la implementación de la arquitectura en GCP (Google Cloud Platform), puesto que la SparkSession ya viene configurada al activarse el Kernel PySpark

In [ ]:
# importación de librerias necesarias
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import findspark
import sys
from datetime import datetime
import pandas as pd

In [ ]:
# Ayuda a Python a encontrar la instalación local de Apache Spark
findspark.init()

In [ ]:
# Definición de ruta que tomará Spark donde escribirá cualquier Data Warehouse manejado (con tablas manejadas) y sus metadatos, que se cree en esta SparkSession
os.environ["SPARK_SQL_WAREHOUSE_DIR"] = "C:/tmp/spark-warehouse" # Aunque ciertamente no es obligatorio en este trabajo

In [ ]:
# Constructor de SparkSession con Delta y Kafka
builder = (
    SparkSession
     .builder
     .appName("Kafka+DeltaInNotebook")
     # Estas dos líneas habilitan las extensiones Delta Lake
     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
     # Se descarga e incorpora el conector Delta Lake JAR a través de Maven
     .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.0")
     # Los conectores de Spark para Kafka se han descargado de Maven y guardado localmente. Se leen desde el directorio local. 
     .config("spark.jars",
             r"C:\Users\TPG\Documents\Spark\spark-3.5.3-bin-hadoop3\spark-3.5.3-bin-hadoop3\jars\spark-sql-kafka-0-10_2.12-3.5.3.jar,"
             r"C:\Users\TPG\Documents\Spark\spark-3.5.3-bin-hadoop3\spark-3.5.3-bin-hadoop3\jars\kafka-clients-3.4.1.jar,"
             r"C:\Users\TPG\Documents\Spark\spark-3.5.3-bin-hadoop3\spark-3.5.3-bin-hadoop3\jars\spark-token-provider-kafka-0-10_2.12-3.5.3.jar,"
             r"C:\Users\TPG\Documents\Spark\spark-3.5.3-bin-hadoop3\spark-3.5.3-bin-hadoop3\jars\commons-pool2-2.11.1.jar"
            )
     # Se utiliza RawLocalFileSystem para escrituras locales, de modo que se evite la creación de ficheros CRC acompañando los ficheros parquet con contenido duplicado.
     .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem")
)

In [ ]:
# Creación de SparkSession con Delta y Kafka
print("Creando la SparkSession...")
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("-"*70)
if spark and not spark.sparkContext._jsc.sc().isStopped():
    creation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"SparkSession creada correctamente a las {creation_time}")
    print(f"Versión de Spark: {spark.version}")
    print(f"Master: {spark.sparkContext.master}")
else:
    print("No se pudo crear la SparkSession o el SparkContext está detenido.")

## 1. Proceso de lectura desde Spark usando *Kafka* como fuente de datos

**1.1.** Se especifican las configuraciones del cliente para consumir mensajes del tópico call-event

In [ ]:
# ¿Qué timezone usa Spark SQL?
print("spark.sql.session.timeZone =", spark.conf.get("spark.sql.session.timeZone"))

In [ ]:
# ¿Qué timezone tiene la JVM/executors?
print("JVM user.timezone =", spark.sparkContext._jvm.java.lang.System.getProperty("user.timezone"))

In [ ]:
# Esta configuración garantiza transformaciones wide más eficientes con volúmenes pequeños de datos
spark.conf.set("spark.sql.shuffle.partitions", 16)

# Se configura la zona horaria de entrada para evitar que campos de tipo fecha sean formateados en una zona horaria distinta a la de interés alterando la fecha real
spark.conf.set("spark.sql.session.timeZone", "America/Bogota")

# Define las opciones principales de Kafka tomando la información presente en la configuración del cliente Kafka
kafka_options = {
    "kafka.bootstrap.servers": "pkc-lgk0v.us-west1.gcp.confluent.cloud:9092",
    "subscribe": "call-event",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'org.apache.kafka.common.security.plain.PlainLoginModule required '
        'username="XYF75DHJY4VF6GYH" '
        'password="cflt1eQHXmINZarEldpn2vXpFgWLVftptFondbaRigAh79IvgQiQQIYEUCmr4i8A";'
    ),
    "kafka.session.timeout.ms": "45000",
    "kafka.client.id": "ccloud-python-client-944d91cd-1a1b-4f4c-8015-c3351aae74a5"
}

**1.2.** Se inicia el proceso de lectura con las configuraciones anteriores

In [ ]:
# Crea un DataFrame de streaming leyendo de Kafka
tam_lote = 3
callStreamingDF = (spark
    .readStream
    .format("kafka")
    .options(**kafka_options)
    .option("maxOffsetsPerTrigger", tam_lote) # Util para garantizar que por microbatch, se lean los tam_lote mensajes de la cola de Kafka
    .option("startingOffsets", "earliest")  # Indica donde comenzar la lectura cuando el proceso de lectura inicia por primera vez. Importante: si ya existe un checkpointLocation con offsets guardados, Spark ignora startingOffsets y resume desde los offsets en el checkpoint. 
    .load() # Con load() nos suscribimos al tópico call-event y empezamos a leer mensajes de el.
)

In [ ]:
# Mostramos el esquema para verificar una correcta conexión al tópico
callStreamingDF.printSchema()

**1.3.** El Streaming Dataframe leído debe formatearse para conseguir el Streaming Dataframe deseado únicamente con los campos necesarios de los eventos de llamadas

In [ ]:
# Importamos las librerias necesarias
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType

In [ ]:
# Definimos el esquema con el que se estructurarán las cadenas JSON deserializadas
esquema_formateo_msg_calls = StructType([
  StructField("call_date", StringType(), nullable=False), # Aunque es cierto que debería formatearse a tipo fecha, existe una deriva de formatos que necesita tratarse en un momento posterior a la lectura.
  StructField("full_name", StringType(), nullable=True),
  StructField("list_id", StringType(), nullable=True),
  StructField("gmt_offset_now", DoubleType(), nullable=True),
  StructField("comments", StringType(), nullable=False),
  StructField("length_in_sec", DoubleType(), nullable=True), # Aunque el campo es realmente entero en la muestra de llamadas, debe leerse inicialmente como flotante para evitar que Spark lo interprete como nulo
  StructField("alt_dial", StringType(), nullable=True),
  StructField("list_name", StringType(), nullable=False),
  StructField("status_name", StringType(), nullable=True),
  StructField("custom_fields", StringType(), nullable=True),
  StructField("Yard Sign", StringType(), nullable=True),
  StructField("Top_Issue", StringType(), nullable=True),
  StructField("New Calls", StringType(), nullable=True)
])

A partir del Streaming Dataframe leído de Kafka, mantenemos únicamente los campos de interés, en la mayoría de los casos, solamente el campo value, lo convertimos al tipo de dato String, obteniéndose un Streaming Dataframe de cadenas JSON. A continuación, cada cadena se estructura a través del esquema definido en la celda anterior, y se consigue un campo de objeto tipo struct (estructura formada por los campos y valores de las cadenas JSON, de forma similar a una tupla). Gracias a la tipología del campo, es posible acceder a los valores de cada tupla mediante el operador punto y generar los campos del esquema. Se aprovecha también esta etapa de formateo para renombrar los campos a nombres estandar para la empresa sin espacios ni caracteres problemáticos. En el código siguiente se materializa lo mencionado:

In [ ]:
parsedCallsKafkaDF = (callStreamingDF
     # Seleccionar únicamente el campo value , convertirlo a string, y estructurarlo según el esquema.       
     .select("value")
     .withColumn("value", F.col("value").cast(StringType()))
     .withColumn("tuplas", F.from_json(F.col("value"), esquema_formateo_msg_calls))
     # expandir la struct a columnas separadas
     .select("tuplas.*")
     .withColumn(
        "call_date",
         F.to_timestamp(F.col("call_date"), "MM/dd/yyyy HH:mm")
     )
)

**1.4.** Ahora que el Streaming ha sido formateado, es hora de escribirlo en la capa Bronze de llamadas en formato Parquet. Dado que este proceso finaliza este Spark Job, es necesario implementar la acción **start()** de modo que se cree un flujo ETL cíclico que se mantenga sondeando las particiones del tópico **call-event** una vez se haya finalizado la escritura en la capa Bronze de llamadas, de las particiones del microbatch construído en cada lote leído. 

Cada microbatch debe filtrarse según el nombre de candidato y escribirse en la carpeta correspondiente. Veámoslo:

In [ ]:
# Función para escribir cada microbatch por candidato
bronze_root = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze" # Definición de ruta Bronze (Se cambia a la del bucket de GCS)
checkpoint_location_bronze_calls = f"C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center Prueba/Checkpoints/bronze_calls_checkpoint"

def write_by_candidate(batch_df, batch_id):
    # Particiones que Spark creó para el DataFrame (RDD partitions)
    spark_parts = batch_df.rdd.getNumPartitions()
    # Mostramos el total de particiones del RDD
    print(f"microbatch {batch_id}: spark_partitions={spark_parts}")

    # Se cachea el microbatch porque se utiliza continuamente durante el ciclo de candidatos
    batch_df.persist() 
    # obtenemos candidatos únicos
    candidates = [row.list_name for row in batch_df.select("list_name").distinct().collect()]
    for cand in candidates:
        out_path = f"{bronze_root}/{cand}/Calls"
        (batch_df
            .filter(F.col("list_name") == cand)
            .repartition(1)
            .sortWithinPartitions(F.col("call_date").asc()) # Transformación para ordenar microbatch por el campo de fecha de forma ascendente (operación Narrow costosa) 
            .write
            .mode("append")
            .parquet(out_path)      # o .format("delta").save(out_path)
        )
    batch_df.unpersist() # Una vez escrito el microbatch completo se libera memoria porque no se necesita posteriormente

In [ ]:
# Se inicia el proceso de streaming con foreachBatch
print("Proceso Streaming ejecutándose...")
query_elt_bronze_calls = 
(
    parsedCallsKafkaDF.writeStream
        .foreachBatch(write_by_candidate)
        # Se almacena un checkpoint de los mensajes leídos de cada partición del tópico call-event para reanudar lecturas sin reprocesamiento de mensajes
       # .option("checkpointLocation", checkpoint_location_bronze_calls) 
        .start()
        #.awaitTermination() # bloquea el hilo actual hasta que la query de streaming termine. Desactivar para ejecutar en segundo plano
)

**Estado**: El código se ha ejecutado y funciona correctamente
**Observaciones**: Aunque se han generado 4 ficheros (1 por batch donde se leen 3 mensajes de 12 mensajes totales en el tópico), cada uno no contiene 3 registros como se esperaba. También cada fichero parquet escrito viene acompañado de un fichero CRC con el mismo nombre.

(LO SIGUIENTE NO APARECERÁ EN EL NOTEBOOK FINAL)

## Verificación de escritura exitosa de eventos de llamada presente en el tópico call-event de Kafka

Verifiquemos que los ficheros parquet escritos tengan los datos correctos

In [ ]:
# Importación de librerias necesarias
import pandas as pd

In [ ]:
# Definición de paths de ficheros
path_1 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze\candidate_name_3\Calls\part-00000-d687f915-82f4-46da-bc8e-7c6f5a163a2d-c000.snappy.parquet"
path_2 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze\candidate_name_3\Calls\part-00000-2415aaf8-6961-4675-8c18-bfb427d03013-c000.snappy.parquet"
path_3 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze\candidate_name_3\Calls\part-00000-77d63018-6ad3-4082-90e7-76e5b64efdc5-c000.snappy.parquet"
path_4 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze\candidate_name_3\Calls\part-00000-a7fc9dc0-c20f-461d-a13d-f2400caaf5da-c000.snappy.parquet"

In [ ]:
# Cargamos en memoria los datos de los ficheros
df_1 = pd.read_parquet(path_1)
df_2 = pd.read_parquet(path_2)
df_3 = pd.read_parquet(path_3)
df_4 = pd.read_parquet(path_4)

In [ ]:
# Verificamos las tipologías
display(df_1.info(), df_2.info(), df_3.info(), df_4.info())

In [ ]:
# Mostramos el contenido de los Dataframes
display(df_1, df_2, df_3, df_4)

**Notas sobre lectura de Kafka**: Spark recorre en la lectura todas las particiones del tópico donde existen datos. Así en general se espera que cada microbatch (RDD) contenga un total de particiones igual al del tópico de Kafka. Se tiene un conocimiento absoluto acerca de cuántas particiones se crean en cada microbatch. El número de particiones sondeadas y de donde se extraen datos, es igual al número de particiones de cada microbatch ensamblado.

#### ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Construcción y carga de tablas Delta para votantes y llamadas

En el documento del TFM desarrollado, este Job de Spark se corresponde con el paso metodológico **2.** *Extracción, transformación, y almacenamiento automatizado de ficheros Bronze en la capa Silver*. El objetivo será sensar la capa bronze tanto de llamadas como de votantes para **extraer** ficheros de votantes y llamadas, transformarlos de modo que trasmuten a un estado curado, y posteriormente depositarlos en formato Delta con particionamiento en la capa Silver correspondiente. Revise la **figura 5** en el documento del TFM para mayor claridad. Recordar también que las tablas Delta discutidas en el documento TFM permiten consultas analíticas OLAP y consultas transacciones OLTP. Los pasos que siguen permiten ir de las musas al teatro: 

## 1. TABLA DELTA DE LLAMADAS

Previo a empezar el flujo ETL incremental que alimenta la tabla Delta de llamadas, es una buena práctica inicializarla de modo que se fije un esquema que debe seguirse de forma riguroso. En caso contrario, la tabla Delta no permitirá inserción de registros.

### 1.1. Inicialización de tabla Delta de llamadas

Este paso es simple: Inicialmente se establece el **esquema** de la tabla Delta a través de un objeto de tipo **StructType**, donde se indicará el nombre estandarizado de cada campo, se indicará su tipología adecuada, y se establecerá si el campo puede o no contener valores nulos. Seguidamente se crea un Dataframe de Spark vacío con el esquema creado y se persiste en modo de sobre-escritura y en formato Delta en la capa Silver de llamadas, con particionamiento por el campo **candidate_name**, a través de su ruta asociada.

In [ ]:
spark.conf.set("spark.sql.caseSensitive", "true")

In [ ]:
from pyspark.sql.types import TimestampType

In [ ]:
# Esquema de tabla Delta (representa el esquema buscado por la empresa)
esquema_delta_calls = StructType([
    StructField("call_date",           TimestampType(), nullable=False),
    StructField("day_of_the_week",     StringType(), nullable=True),
    StructField("time_slot",           StringType(), nullable=True),
    StructField("operator_name",       StringType(), nullable=True),
    StructField("list_id",             StringType(), nullable=True),
    StructField("gmt_offset_now",      DoubleType(), nullable=True),
    StructField("voter_id",            StringType(), nullable=False),
    StructField("length_in_sec",       IntegerType(), nullable=True),
    StructField("strategy_dial",      StringType(), nullable=True),
    StructField("candidate_name",      StringType(), nullable=False),  # clave de partición con el nombre del candidato de la campaña
    StructField("initial_call_status_name",      StringType(), nullable=True),
    StructField("final_call_status_name",      StringType(), nullable=True),
    StructField("yard_sign",           StringType(), nullable=True),
    StructField("top_issue",           StringType(), nullable=True),
    StructField("campaign",            StringType(), nullable=True),
    StructField("converted",           StringType(), nullable=True)    
]
)

In [ ]:
# Vamos a crear la carpeta Delta (/_delta_log/, metadata, etc.)
# con la definición de esquema completa, particionada por "candidate_name".
# Para ello creamos un DataFrame vacío con ese esquema y lo persistimos:

# Ruta de salida en local (cambiar a la ruta del bucket de GCS)
#silver_calls_path = "C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center Prueba/Silver/Calls"
silver_calls_path = "C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center All/Silver/Calls" # Silver path calls All

# Creamos un DataFrame vacío con el schema_completo
df_calls_vacio = spark.createDataFrame(
    spark.sparkContext.emptyRDD(), 
    esquema_delta_calls # Este esquema hace referencia al esquema que se desea obtener con los nombres de campos estándar para la empresa. Aquí se incluyen las etiquetas de los clústeres de grupos de nombres de campos
)

In [ ]:
# Lo escribimos en modo overwrite para inicializar la tabla Delta
(
    df_calls_vacio.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("candidate_name")
        .save(silver_calls_path)
)

# Mostramos que la tabla Delta se ha inicializado correctamente 
print("Inicialización completada: tabla Delta de llamadas creada correctamente.\nProceda con el proceso ETL incremental para la carga de la tabla.")

**Nota**: Este código debe ejecutarse una única vez, a menos que el esquema deba ser actualizado modificando el actual o agregando campos nuevos que se necesiten para el tratamiento de datos particular de interés.

### 1.2. Carga de la tabla Delta de llamadas vía un Job de Spark para ETL incremental

Es hora de rastrear a través de Spark Structured Streaming la capa Bronze de llamadas y ejecutar un proceso ETL incremental de tiempo real para cada fichero que se va añadiendo también en tiempo real, a través del Job de Spark ETL de llamadas creado en el paso metodológico **1.b**. A continuación se divide el proceso seguido en las 4 etapas: **extracción**, **reducción e integración**, **limpieza y transformación**, y **carga de datos**:

#### Extracción 

Se realiza una lectura en tiempo real de los ficheros de llamadas que el Job de Spark anterior escribe también en tiempo real. Por supuesto ejecutar la celda siguiente no materializa cada fichero leído en memoria, puesto que solo se implementan transformaciones que por definición son **lazy**. Simplemente serán registradas las operaciones en el DAG de ejecución de Spark que vive en el Driver del clúster de Spark

In [ ]:
# El esquema siguiente está conciliado con el esquema de los ficheros parquet de la capa Bronze de llamadas
bronze_schema_calls = StructType([
  #StructField("call_date", TimestampType(), nullable=False), # Aunque es cierto que debería formatearse a tipo fecha, existe una deriva de formatos que necesita tratarse en un momento posterior a la lectura.
  StructField("call_date", StringType(), nullable=False), 
  StructField("full_name", StringType(), nullable=True),
  #StructField("list_id", StringType(), nullable=True), # Desmarcar cuando se lean ficheros de llamdas que han sido escritos con Kafka
  StructField("list_id", LongType(), nullable=True),
  StructField("gmt_offset_now", DoubleType(), nullable=True),
  StructField("comments", StringType(), nullable=False),
  StructField("length_in_sec", DoubleType(), nullable=True), # Aunque el campo es realmente entero en la muestra de llamadas, debe leerse inicialmente como flotante para evitar que Spark lo interprete como nulo
  StructField("alt_dial", StringType(), nullable=True),
  StructField("list_name", StringType(), nullable=False),
  StructField("status_name", StringType(), nullable=True),
  StructField("custom_fields", StringType(), nullable=True),
  StructField("Yard Sign", StringType(), nullable=True),
  StructField("Top_Issue", StringType(), nullable=True),
  StructField("New Calls", StringType(), nullable=True)
])

In [ ]:
# Ruta dinámica asociada a la capa Bronze de llamadas
# path_bronze_calls = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze\*\Calls\*.parquet" # Uso de wildcard para todas las campañas
path_bronze_calls = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center All\Bronze\Calls\*.parquet" # Path Bronze Calls All

# Se definen las transformaciones que han de leer ficheros de la capa Bronze
# de llamadas, y cargarlos en la memoria RAM de los executors de forma
# particionada con el esquema especificado.
streamingCallsBronzeDF = (
    # Naturalmente el esquema difiere del esquema Delta porque este debe estar
    # conciliado con el esquema embebido en los ficheros Parquet de la capa Bronze
    # de llamadas, que no está curado.
    spark.readStream
        .schema(bronze_schema_calls)
        # De este modo se garantiza que por cada paso del flujo cíclico de ETL
        # incremental se lea, procese y escriba solamente un fichero.
        # .option("maxFilesPerTrigger", 1)
        .parquet(path_bronze_calls)
)

#### Reducción e integración (Se omite porque no tiene sentido incluirlo en este caso de llamadas. En votantes por supuesto si se incuirá)

#### Limpieza y transformación

Esta etapa es crítica y determina en gran medida la calidad final de los datos que serán consumidos principalmente para consultas analíticas y entrenamiento de modelos de Machine Learning. Tiene por objetivo curar los datos crudos de la capa Bronze de llamadas y crear nuevos atributos a partir de los atributos originales. A través de una metodología genérica definida en el documento del TFM, se guía la construcción del código necesario para la ejecución de esta etapa. 

##### Importación de librerias necesarias para la limpieza de datos

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType

##### **Paso 1:** Supresión de cadenas invisibles en campos de tipo string

In [ ]:
# El pipeline de transformación que se ha creado, está alineado con la metodología documentada en el TFM para la fase de limpieza y transformación de los datos de llamadas de Bronze. 
# ------------ PASO 1: Eliminación de carácteres invisibles --------------------------------
# Descripción: expresión de texto normalizada (eliminar control/anchura cero, recortar, colapsar o eliminar espacios)

def suprimir_caracteres_invisibles(col):
    # Reemplazar los caracteres de control, los de ancho cero y NBSP por un espacio normal (evita que se unan valores con más de una palabra separadas por espacio).
    cleaned = F.regexp_replace(
        col.cast(StringType()),
        r'[\u0000-\u001F\u007F\u180E\u200B\u2060\uFEFF\u00A0]',
        ' '
    )
    # Se trimea y colapsa cualquier secuencia de espacios en blanco (tabulaciones/saltos de línea/espacios múltiples) a un solo espacio normal.
    collapsed = F.regexp_replace(F.trim(cleaned), r'\s+', ' ')
    return collapsed

# Función para imputar valores vacíos, en blanco o nulos con el rótulo "NULL"
def imputar_null_literal(expr_col):
    return F.when((expr_col.isNull()) | (F.trim(expr_col) == ""), F.lit("NULL")).otherwise(expr_col)

# Ciclo con mapeo (legible y fácil de mantener)
# -------------------------
# Mapeo: columna origen -> (tipo_transformación, nombre_destino)
# Tipos de transformación = "title" (initcap(lower)), "upper" (upper), "None" (sin tranformación)
mappings = {
    "full_name": ("title", "operator_name"),
    "alt_dial": ("title", "strategy_dial"),
    "status_name": ("title", "initial_call_status_name"),
    "custom_fields": ("title", "final_call_status_name"),
    "Yard Sign": ("title", "yard_sign"),
    "Top_Issue": ("title", "top_issue"),
    "New Calls": ("upper", "campaign"),
    "list_name": ("None", "candidate_name")
    # list_id sólo se imputa (no normalizamos)
}

# df = streamingCallsBronzeDF Ejecutar cuando se lean ficheros que se escriben leyendo de Kafka y comentar las líneas del dataframe inmediatamente inferior

df = (
streamingCallsBronzeDF
        .withColumn(
        "call_date",
         F.to_timestamp(F.col("call_date"), "M/d/yyyy H:mm")
        )
)

for src_col, (transform_type, dest_col) in mappings.items():
    # Paso 1: Normalización: suprimir_caracteres_invisibles(...)  (si es UDF o función de columna)
    normalized = suprimir_caracteres_invisibles(F.col(src_col))
    # Pasos 2 y 3: Conversión de campos requeridos a formato title y Upper
    if transform_type == "title":
        cased = F.initcap(F.lower(normalized))
    elif transform_type == "upper":
        cased = F.upper(normalized)
    else:
        cased = normalized

    # Pasos 4 y 7: Imputación de valores nulos en campos categóricos nullables
    # Se aprovecha este paso de imputación de valores nulos para renombrar los campos del Streaming Dataframe a los deseados.
    df = df.withColumn(dest_col, imputar_null_literal(cased))

# imputar list_id (si quieres forzar literal "NULL" cuando es nulo o vacío)
df = df.withColumn("list_id", imputar_null_literal(F.col("list_id").cast(StringType())))

# df ya contiene las columnas nuevas por lo que se eliminan las columnas con nombre antiguo
cols_orig_a_eliminar = list(mappings.keys()) 
streamingCBNoNullDF = df.drop(*cols_orig_a_eliminar)

##### **Pasos 5:** Estandarización de valores de campos

Los campos *initial_call_status_name* y *top_issue* requieren esta transformación al contener valores no estándar que pertenece a alguna de las demás categorías del campos

In [ ]:
# ------ Diccionarios de normalización ------ 
# Mapeador de initial_call_status_name
status_map = {
    "Brock Myers": "Not Supporting",
    "Christopher Harrison": "Not Supporting",
    "Aisha Braveboy": "Not Supporting",
    "Rushern Baker": "Support",
    "Support with Sign": "Support",
    "Declined Sale": "No Answer",
    "Will Send": "Support",
    "Support ONLY": "Support"
}

# Mapeador de top_issue
top_issue_map = {
    "emergencyservices": "Emergency Services",
    "utilitycosts": "Utility Costs",
    "socialservices": "Social Services",
    "morejobs": "More Jobs",
    "rentcontrol": "Rent Control"
}

In [ ]:
# Función de creación de mapas UDF
def map_preserve_original(mapping):
    # Retorna el valor mapeado cuando está presente en el diccionario de mapeo. En caso contrario, mantiene el valor original del campo
    def _mapper(v):
        if v is None:
            return None
        key = v.strip().lower()
        return mapping.get(key, v)   # <-- Se preserva valor original si clave no es encontrada
    return F.udf(_mapper, StringType())

# Creación de mapas UDF
map_status_udf = map_preserve_original(status_map)
map_top_issue_udf = map_preserve_original(top_issue_map)

In [ ]:
# Ejecución de mapeo sobre los campos que lo requieren
streamingCBMappedDF = (streamingCBNoNullDF
        .withColumn("initial_call_status_name", map_status_udf(F.col("initial_call_status_name")))
        .withColumn("top_issue", map_top_issue_udf(F.col("top_issue")))
        )

##### **Paso 6**. Conversión numérica e imputación de nulos

In [ ]:
# ---------- Conversión numérica e imputación de nulos numéricos ----------
# gmt_offset_now se transforma a tipo flotante (campo sin nulos) y length_in_sec a tipo entero (actualmente en tipo flotante)
# Se reemplazan valores nulos por 0 en length_in_sec, indicando la imposibilidad de establecer comunicación con el votante en cuestión
streamingCBCuratedDF = (streamingCBMappedDF
        .withColumn("gmt_offset_now", F.col("gmt_offset_now").cast(DoubleType()))
        .withColumn("length_in_sec",
                   F.coalesce(F.col("length_in_sec").cast(IntegerType()), F.lit(0)).cast(IntegerType()))
)

# ----------- Proyección final para asegurar la presencia de las columnas correctas en el Streaming DataFrame a escribir -----------
streamingCBCuratedDF = streamingCBCuratedDF.select(
    F.col("call_date"),
    F.col("operator_name"),
    F.col("list_id"),
    F.col("gmt_offset_now"),
    F.col("comments").alias("voter_id"),
    F.col("length_in_sec"),
    F.col("strategy_dial"),
    F.col("candidate_name"), # Es necesario renombrar puesto que en la etapa de imputación de nulos no se incluyó este campo obligatorio
    F.col("initial_call_status_name"),
    F.col("final_call_status_name"),
    F.col("yard_sign"),
    F.col("top_issue"),
    F.col("campaign")
)

##### **Paso de transformacion**: Generación de campos derivados **day_of_the_week**, **time_slot**, y **converted**

En este paso de transformación, se generan los campos calculados:

- *day_of_the_week*, que indica el día de la semana de llamada y que será usado en el entrenamiento de modelos de Machine Learning para guiar la construcción de estrategias temporales de llamada eligiendo el día óptimo de llamada.
- *time_slot*, que define la franja horaria de llamada **maniana**, **mediodia**, **tarde**, y **noche** y que será usado en el entrenamiento de modelos de Machine Learning para guiar la construcción de estrategias temporales de llamada eligiendo la franja horaria adecuada de llamada en el día óptimo de llamada.
- *converted*, que será usada como variable a explicar o variable objetivo, clase o dependiente en el modelo de Machine Learning que se construirá más adelante

In [ ]:
# Conjunto de status de llamada que cuenta como conversión
converted_set = {
    "Not interested",
    "Call Back",
    "Already Voted",
    "Spanish Speaker",
    "Busy",
    "Foreign Language",
    "Remove From List",
    "Do Not Call",
    "No Response",
    "Moved",
    "Support",       
    "Considering",
    "Undecided",
    "Not Supporting"
}

streamingCBCuratedDF = (
    streamingCBCuratedDF
    .withColumn(
        "converted",
        F.when(F.col("initial_call_status_name").isin(*sorted(converted_set)), F.lit("Yes"))
         .otherwise(F.lit("No"))
    )
    # Generación de campo inferido de día de la semana a partir de los valores de fecha de llamada. Se retornan valores como "Monday", "Tuesday",...
    .withColumn("day_of_the_week", F.date_format(F.col("call_date"), "EEEE"))
    # Se extrae en una nueva columna, la hora de llamada que servirá para obtener la franja horaria
    .withColumn("hour_of_day", F.hour(F.col("call_date")))
    # Según el rango al que pertenezca la hora deducida, se define la franja horaria:
    .withColumn(
        "time_slot",
        F.when((F.col("hour_of_day") >= 5) & (F.col("hour_of_day") <= 11), F.lit("morning"))
         .when((F.col("hour_of_day") == 12), F.lit("noon"))
         .when((F.col("hour_of_day") >= 13) & (F.col("hour_of_day") <= 17), F.lit("afternoon"))
         .when((F.col("hour_of_day") >= 18) | (F.col("hour_of_day") <= 4), F.lit("evening"))
         .otherwise(F.lit(None))
    )
    # Se remueve la columna de ayuda "hour_of_day"
    .drop("hour_of_day")
)

##### **Paso extra**: Colapso de particiones del RDD leído desde la capa Bronze de llamadas en una única partición 

Como en el Job de Spark para ELT de carga de la capa Bronze de llamadas, puesto que los ficheros parquet son pequeños, previo a la escritura de los Streaming Dataframes curados
se utiliza la transformación wide repartition(1) para consolidar las particiones del RDD miembro del Dataframe en una única partición, de modo que se escriba un único fichero
en la capa Silver. Así, el Spark Job que se desarrolla ejecuta un mapeo 1-1 de los ficheros de la capa Bronze de llamadas. 

In [ ]:
FORCE_SINGLE_OUTPUT_FILE = True   # Se define en True solo para pruebas pequeñas
if FORCE_SINGLE_OUTPUT_FILE:
    streaming_call_to_write_delta = streamingCBCuratedDF.coalesce(1)
else:
    # repartition por la columna de partición ayuda a agrupar datos del mismo us_congress
    streaming_call_to_write_delta = streamingCBCuratedDF.repartition(F.col("candidate_name"))

In [ ]:
# --------------- Escritura de microbatch en el lago de datos y actualización de metadatos de la tabla Delta de llamadas ---------------
# Definición de variables necesarias
# Importación de librerias necesarias
from dotenv import load_dotenv
import requests
from urllib.parse import urlparse, urlunparse
from requests.exceptions import RequestException, SSLError

# Sesión global (no usar proxies del entorno)
_session = requests.Session()
_session.trust_env = False
_session.proxies = {"http": None, "https": None}

_cached_login_token = None

# Función para asegurar solicitud http
def ensure_http(base):
    p = urlparse(base)
    if p.scheme.lower() != "http":
        p = p._replace(scheme="http")
        return urlunparse(p)
    return base
    
# Función de obtención de token de Dremio
def login_get_token(DREMIO_USER, DREMIO_PASSWORD, DREMIO_BASE):
    global _cached_login_token
    base = ensure_http(DREMIO_BASE)
    url = base.rstrip("/") + "/apiv2/login"
    print(f"[DEBUG] login_get_token -> usando URL: {url}")
    payload = {"userName": DREMIO_USER, "password": DREMIO_PASSWORD}
    try:
        r = _session.post(url, json=payload, timeout=10)
        r.raise_for_status()
    except SSLError as e:
        # Mensaje específico para explicar el mismatch TLS/HTTP
        raise RuntimeError(
            "Error SSL durante login. Probablemente la URL está usando https en un puerto que solo acepta http. "
            "Revisa DREMIO_BASE y fuerza http. Detalle: " + str(e)
        )
    except RequestException as e:
        raise RuntimeError(f"Login failed: {e}")
    try:
        data = r.json()
    except Exception:
        raise RuntimeError(f"Login returned non-JSON response: {r.status_code} {r.text}")
    token = data.get("token")
    if not token:
        raise RuntimeError(f"No token in login response: {data}")
    _cached_login_token = token
    return token

# Cargar el archivo .env explícitamente
load_dotenv(dotenv_path="secret_keys.env")

# Claves secretas de conexión a Dremio
DREMIO_BASE = os.getenv("DREMIO_BASE")
DREMIO_USER = os.getenv("DREMIO_USER")
DREMIO_PASSWORD = os.getenv("DREMIO_PASSWORD")
DREMIO_PAT =  _cached_login_token or login_get_token(DREMIO_USER, DREMIO_PASSWORD, DREMIO_BASE)

# Dataset de votantes en Dremio
DREMIO_DATASET_CALLS = "local_delta.Calls" 

In [ ]:
# Definición de funciones necesarias
# Función extra para construir consulta SQL que actualiza metadatos de la tabla Delta de votantes en Dremio 
def build_partition_sql(dataset, key, val):
    comps = dataset.split(".")
    quoted = ".".join(f'"{c}"' for c in comps)
    val = str(val).replace("'", "''")
    return f'ALTER TABLE {quoted} REFRESH METADATA FOR PARTITIONS ({key} = \'{val}\')'

# Función extra para ejecutar consulta SQL que actualiza metadatos de la tabla Delta de votantes en Dremio 
def post_sql(sql):
    url = ensure_http(DREMIO_BASE).rstrip("/") + "/api/v3/sql"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {DREMIO_PAT}"}
    resp = requests.post(url, json={"sql": sql}, headers=headers, timeout=30, verify=False)
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Función para escribir Streaming Dataframe y actualizar metadatos de la tabla Delta de llamadas
def escribrir_delta_actualizar_metadata_calls(batch_df, batch_id):
    batch_df.write.format("delta") \
            .mode("append") \
            .partitionBy("candidate_name") \
            .save(silver_calls_path)
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    # ------- Código de actualización de metadatos------- 
    # Identificar particiones afectadas a través de la extracción de los valores distintos de la clave de partición en el Streaming Dataframe leído. 
    partitions = [row["candidate_name"] for row in batch_df.select("candidate_name").distinct().collect()]

    # 3) llamar Dremio por partición
    for p in partitions:
        sql = build_partition_sql(DREMIO_DATASET_CALLS, "candidate_name", p)
        resp = post_sql(sql)
        print("Dremio response:", resp)

#### Carga de datos

En esta etapa, cada Streaming Dataframe limpiado y transformado deberá escribirse en disco a la fuente o sumidero dentro de la capa Silver de llamadas. Las particiones de dicho Dataframe se escriben en disco desde la memoria RAM de los executors del clúster de Spark. Esto dilucida la razón de generarse tantos ficheros como particiones tenga el RDD miembro del Dataframe, en el proceso de escritura. 


In [ ]:
# Ruta de almacenamiento de checkpoint para evitar reprocesamiento y/o omisión de procesamiento de ficheros cuando se reanude, reinice o falle el Job de Spark
# que sea crea en este Notebook de Jupyter
# checkpoint_path_bronze_calls_processed = f"C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center Prueba/Checkpoints/silver_calls_checkpoint" 
checkpoint_path_bronze_calls_processed = f"C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center All/Checkpoints/silver_calls_checkpoint"

# Se escribe el dataframe de llamadas curado 
query_etl_silver_calls = (
    streaming_call_to_write_delta
        .writeStream
        .foreachBatch(escribrir_delta_actualizar_metadata_calls)
        .option("checkpointLocation", checkpoint_path_bronze_calls_processed) 
        .trigger(once=True) # Se procesan todos los archivos disponibles y se finaliza el flujo ETL (Ideal para flujos ETL sobre ficheros que arriban esporadicamente)
        .start()
        .awaitTermination() # Desactivar para ejecutar en segundo plano. # Importante activarlo para depurar código

)

**Nota**: El pipeline de transformación se ha probado y ha quedado correctamente escrito. Aún falta verificar con más datos. !EXITOS EN TU TRABAJO¡

#### Verificación de limpieza


In [ ]:
import pandas as pd

In [ ]:
# Definición de paths de ficheros
path_1 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Calls\candidate_name=candidate_name_3\part-00000-f431d8a0-fc89-4092-88ea-195d8b01d6ce.c000.snappy.parquet"
path_2 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Calls\candidate_name=candidate_name_3\part-00000-cb0114e5-9f4e-4175-a76a-100e9d7c6bf3.c000.snappy.parquet"
path_3 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Calls\candidate_name=candidate_name_3\part-00000-fbf545b0-ac9f-484e-ab98-fa9132fff2b2.c000.snappy.parquet"
path_4 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Calls\candidate_name=candidate_name_3\part-00000-c5c47f07-7d30-42b1-849d-6b855602d492.c000.snappy.parquet"

In [ ]:
df_1 = pd.read_parquet(path_1)
df_2 = pd.read_parquet(path_2)
df_3 = pd.read_parquet(path_3)
df_4 = pd.read_parquet(path_4)

In [ ]:
df_1.info()

In [ ]:
display(df_1, df_2, df_3, df_4)

#### ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 2. TABLA DELTA DE VOTANTES

Previo a empezar el flujo ETL incremental que alimenta la tabla Delta de votantes, es una buena práctica inicializarla de modo que se fije un esquema que debe seguirse de forma riguroso y que será el ideal al que se desea llegar. En caso contrario, la tabla Delta no permitirá inserción de registros.

### 2.1. Inicialización de tabla Delta de votantes

El script que sigue permitirá inicializar la tabla Delta del mismo modo que en los datos de llamadas. Se indica el campo **us_congress** como campo de particionamiento y se especifica el path asociado a la capa Silver de votantes

In [ ]:
spark.conf.set("spark.sql.caseSensitive", "true")

In [ ]:
# Esquema de tabla Delta de votantes (representa el esquema buscado por la empresa)
esquema_delta_voters = StructType([
    StructField("voter_id",           StringType(), nullable=False),
    StructField("genre",       StringType(), nullable=True),
    StructField("age",             IntegerType(), nullable=True),
    StructField("us_congress",      StringType(), nullable=False),  # clave de partición con el número de dsitrito del congreso de US.
    StructField("city",            StringType(), nullable=True),
    StructField("state",       StringType(), nullable=True),
    StructField("st_senate",      StringType(), nullable=True),
    StructField("house_district",      StringType(), nullable=True), 
    StructField("sboe",      StringType(), nullable=True),
    StructField("party",      StringType(), nullable=True),
    StructField("ethnic_description",           StringType(), nullable=True)
]
)

In [ ]:
# Vamos a crear la carpeta Delta (/_delta_log/, metadata, etc.)
# con la definición de esquema delta, particionada por "us_congress".
# Para ello creamos un DataFrame vacío con ese esquema y lo persistimos:

# Ruta de salida en local (cambiar a la ruta del bucket de GCS)
# silver_voters_path = "C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center Prueba/Silver/Voters"
silver_voters_path = "C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center All/Silver/Voters" # Silver Voters Path All

# Creamos un DataFrame vacío con el schema_completo
df_voters_vacio = spark.createDataFrame(
    spark.sparkContext.emptyRDD(), 
    esquema_delta_voters # Este esquema hace referencia al esquema que se desea obtener con los nombres de campos estándar para la empresa. Aquí se incluyen las etiquetas de los clústeres de grupos de nombres de campos
)

In [ ]:
# Lo escribimos en modo overwrite para inicializar la tabla Delta
(
    df_voters_vacio.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("us_congress")
        .save(silver_voters_path)
)

# Mostramos que la tabla Delta se ha inicializado correctamente 
print("Inicialización completada: tabla Delta de votantes creada correctamente.\nProceda con el proceso ETL incremental para la carga de la tabla.")

**Nota**: Este código debe ejecutarse una única vez, a menos que el esquema deba ser actualizado modificando el actual o agregando campos nuevos que se necesiten para el tratamiento de datos particular de interés.

### 2.2. Carga de la tabla Delta de votantes vía un Job de Spark para ETL incremental

#### Extracción (SÚPER-ESQUEMA) 

Como en la tabla de llamadas, el proceso de extracción consiste en un sondeo de la capa Bronze de votantes para descubrir y leer ficheros de votantes en formato Parquet que se cargan en la memoria RAM de los executors del clúster de procesamiento ante la llamada de la acción start() sobre el Streaming Dataframe de cada fichero de votantes en un estado curado. El primer gran paso, que es fundamental e imprescindible en este Job de Spark, será especificar el **SÚPER-ESQUEMA** que se menciona generosamente en el documento del TFM, como mecanismo para conseguir la integración de los ficheros de votantes con deriva de esquema, en un único Streaming Dataframe. Aunque es simple y presenta limitaciones de escalabilidad, es suficiente para este trabajo. 


In [ ]:
# El esquema siguiente debe estar conciliado con todos los esquemas de los ficheros parquet de la capa Bronze de votantes
# Definir el esquema de ingreso (super-schema) para lecturas desde Bronze
# Incluye variantes de nombre para cada clúster y los campos de interés
bronze_super_schema_voters = StructType([
    StructField("voter_id",        StringType(),  nullable=False),
    StructField("Voters_StateVoterID", StringType(), nullable=False),
    StructField("StateFileID", StringType(), nullable=False),
    StructField("sex", StringType(), nullable=True),
    StructField("Sex", StringType(), nullable=True),
    StructField("Voters_Gender", StringType(), nullable=True),
    StructField("AGE", LongType(), nullable=True),
    StructField("Age", LongType(), nullable=True),
    StructField("Voters_Age", LongType(), nullable=True),
    StructField("US CONGRESS", StringType(),  nullable=False),
    StructField("CD", StringType(), nullable=False),
    StructField("US_Congressional_District", StringType(), nullable=False),
    StructField("city", StringType(), nullable=True),
    StructField("City", StringType(), nullable=True),
    StructField("Residence_Addresses_City", StringType(), nullable=True),
    StructField("state", StringType(), nullable=True),
    StructField("State", StringType(), nullable=True),
    StructField("Residence_Addresses_State", StringType(), nullable=True),
    StructField("ST SENTATE", StringType(),  nullable=True),
    StructField("SD", StringType(), nullable=True),
    StructField("State_Senate_District", StringType(), nullable=True),
    StructField("ST REP", StringType(),  nullable=True),
    StructField("HD", StringType(), nullable=True),
    StructField("State_House_District", StringType(), nullable=True),
    StructField("SBOE", StringType(),  nullable=True),
    StructField("Parties_Description", StringType(),  nullable=True),
    StructField("EthnicGroups_EthnicGroup1Desc", StringType(),  nullable=True)   
])

En la tabla siguiente se relaciona el súper-esquema con el esquema Delta, se especifica la regla de mapeo usada y se escribe por clúster, una nota con características importantes y detalles sobre el proceso de limpieza del conjunto de campos que contiene dicho clúster:

<h5 style="text-align:center">Voter Schema Mapping: Bronze → Silver (Delta)</h5>

| Original Fields                              |        Cluster Name | Type    | Mapping rule                                                     | Notas                                                                |
| -------------------------------------------- | ------------------: | ------- | ---------------------------------------------------------------- | -------------------------------------------------------------------- |
| voter\_id, Voters\_StateVoterID, StateFileID |           voter\_id | String  | `coalesce(voter_id, Voters_StateVoterID, StateFileID)`           | Campo sin errores; Pseudonimizado irreversiblemente por tratarse de un identificador de votantes directo.                 |
| sex, Sex, Voters\_Gender                     |               genre | String  | `coalesce(sex, Sex, Voters_Gender)` | Valores nulos: 3.9% y 5.11% en 2 de 3 campañas. Imputar valores nulos con la etiqueta literal "NULL" (string).    |
| AGE, Age, Voters\_Age                        |                 age | Integer | `coalesce(AGE, Age, Voters_Age).cast(Integer)`                   | Valores nulos en porcentaje insignificante. Reemplazar nulos por la media del campo y redondear a entero si corresponde.                    |
| US CONGRESS, CD, US\_Congressional\_District |        us\_congress | String  | `coalesce("US CONGRESS", "US_Congressional_District", CD)`       | Campo sin errores. Se anonimiza a través de pseudonimización reversible al tratarse de un cuasi-identificador de votantes. No se almacena tabla de correspondencia en la arquitectura.          |
| city, City, Residence\_Addresses\_City       |                city | String  | `coalesce(city, City, Residence_Addresses_City)`                 | Campo con errores de espacios invisibles. Normalizar a través de trim y al tratarse de un cuasi-identificador de votantes, se anonimiza a través de pseudonimización reversible.        |
| state, State, Residence\_Addresses\_State    |               state | String  | `coalesce(state, State, Residence_Addresses_State)`              | Campo sin errores. Al tratarse de un campo de siglas, hacer UPPERCASE para garantizar consistencia. Anonimizar a través de pseudonimización reversible por tratarse de un cuasi-identificador|
| ST SENTATE, SD, State\_Senate\_District      |          st\_senate | String  | `coalesce("ST SENTATE", SD, State_Senate_District)`              | Valores nulos en porcentaje insignificante. Reemplazar nulos por la etiqueta "NULL". Pseudonimizar reversiblemente por ser cuasi-identificador.
| ST REP, HD, State\_House\_District           |     house\_district | String  | `coalesce("ST REP", HD, State_House_District)`                   | Valores nulos en porcentaje insignificante. Reemplazar nulos por "NULL". Pseudonimizar reversiblemente por ser cuasi-identificador.                       |
| SBOE                                         |                sboe | String  | `coalesce(SBOE)`                                                 | Campo sin errores. Pseudonimizar reversiblemente por ser cuasi-identificador. Aparece en 1/3 campañas.                                       |
| Parties\_Description                         |               party | String  | `coalesce(Parties_Description)`                                  | Valores nulos en un porcentaje insignificante. Presencia de errores de espacios invisibles e inconsistencia de mayúsculas/minúsculas. Normalizar a través de trim, y title. Reemplazar nulos por "NULL". Pseudonimizar reversiblemente al tratarse de un campo de carácter personal dentro de las categorías de datos especiales. Aparece en 1/3 campañas.                         |
| EthnicGroups\_EthnicGroup1Desc               | ethnic\_description | String  | `coalesce(EthnicGroups_EthnicGroup1Desc)`                        | Valores nulos: 10.24% en 1/3 campañas. Presencia de errores de espacios invisibles e inconsistencia de mayúsculas/minúsculas. Normalizar a través de trim, y title. Reemplazar nulos por "NULL". Pseudonimizar reversiblemente por tratarse de un dato personal perteneciente a categorías especiales. Solamente aparece en una de las tres campañas                |


Especificado el súper-esquema de lectura y mostrada la tabla que relaciona el super-esquema con el esquema Delta, procedemos a escribir el código de Spark Structured Streaming que vigilará la capa Bronze de votantes y leerá ficheros no procesados, con el **SÚPER-ESQUEMA** construído

In [ ]:
# Ruta dinámica asociada a la capa Bronze de llamadas
# path_bronze_voters = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Bronze\*\Voters\*.parquet" # Uso de wildcard para todas las campañas 
path_bronze_voters = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center All\Bronze\Voters\*\*.parquet" # Path Bronze Voters All

# Se definen las transformaciones que han de leer ficheros de la capa Bronze de votantes, y cargarlos en la memoria RAM de los executors de forma particionada con el esquema especificado
streamingVotersBronzeDF = (
    # Naturalmente el esquema difiere del esquema Delta porque este debe estar
    # conciliado con el esquema embebido en los ficheros Parquet de la capa Bronze
    # de llamadas, que no está curado.
    spark.readStream
    .schema(bronze_super_schema_voters)
    # De este modo se garantiza que por cada paso del flujo cíclico de ETL incremental se lea, procese y escriba solamente un fichero.
    .option("maxFilesPerTrigger", 3)
    .parquet(path_bronze_voters)
)

#### Reducción e integración 

En esta etapa, ha de imaginar el lector un gran Streaming Dataframe de Spark con los campos presentes en el **SUPER-ESQUEMA**. En cada *trigger* o disparador, el Streaming DataFrame resultante tendrá datos en los campos que resultan de la intersección del esquema interno de cada fichero descubierto y leído con el **SÚPER-ESQUEMA**. Los campos presentes en el **SUPER-ESQUEMA** y no en el fichero leído, estarán poblados en su totalidad de valores nulos. A través de la transformación *narrow* *coalesce*, será posible colapasar o combinar cada clúster de columnas en un único campo que heredará el rótulo del clúster. El esquema del Streaming Dataframe obtenido después de la *reducción* debe ser idéntico al esquema con el que se incializó la tabla Delta de llamadas. 

Para que esta fase de reducción e integración funcione adecuadamente, ha de configurarse: 

In [ ]:
spark.conf.set("spark.sql.caseSensitive", "true")

Evitando que campos con nombres estrictamente diferentes como Title y title sean indistinguibles

In [ ]:
# El código siguiente permite ejecutar la etapa de reducción (en la misma lectura con el super-esquema se ha ejecutado el proceso de integración)
# Importación de librerias necesarias
from pyspark.sql import functions as F

# 1) Colapsar columnas redundantes a columnas canonicas
streamingVBReducedDF = streamingVotersBronzeDF.select(
    # voter_id (se captura la primera columna no nula entre las candidatas)
    F.coalesce(F.col("voter_id"), F.col("Voters_StateVoterID"), F.col("StateFileID")).alias("voter_id"),

    # género crudo 
    F.coalesce(F.col("sex"), F.col("Sex"), F.col("Voters_Gender")).alias("genre"),

    # edad (se asegura además que el campo sea entero)
    F.coalesce(F.col("AGE"), F.col("Age"), F.col("Voters_Age")).cast(IntegerType()).alias("age"),

    # clave de partición us_congress
    F.coalesce(F.col("US CONGRESS"), F.col("US_Congressional_District"), F.col("CD")).alias("us_congress"),

    # ciudad y estado
    F.coalesce(F.col("city"), F.col("City"), F.col("Residence_Addresses_City")).alias("city"),
    F.coalesce(F.col("state"), F.col("State"), F.col("Residence_Addresses_State")).alias("state"),

    # senate / house
    F.coalesce(F.col("ST SENTATE"), F.col("SD"), F.col("State_Senate_District")).alias("st_senate"),
    F.coalesce(F.col("ST REP"), F.col("HD"), F.col("State_House_District")).alias("house_district"),

    # otros
    F.coalesce(F.col("SBOE")).alias("sboe"),
    F.coalesce(F.col("Parties_Description")).alias("party"),
    F.coalesce(F.col("EthnicGroups_EthnicGroup1Desc")).alias("ethnic_description"),

)

Efectuada la etapa de *integración* y *reducción*, es apropiado ejecutar las cargas de trabajo de *limpieza* y *transformación*. Como en el caso de los datos de las llamadas, se guía este proceso a través de una metodología genérica que puede revisarse detalladamente en el documento del TFM.

#### Limpieza y transformación

Esta etapa es crítica y determina en gran medida la calidad final de los datos que serán consumidos principalmente para consultas analíticas y entrenamiento de modelos de Machine Learning. Tiene por objetivo curar los datos crudos de la capa Bronze de votantes y crear nuevos atributos a partir de los atributos originales. A través de una metodología genérica definida en el documento del TFM, se guía la construcción del código necesario para la ejecución de esta etapa. Además es buena práctica utilizar **forEachBatch** para procesar cada microbatch como un DataFrame por lotes. La integración no continuada de ficheros de votantes, hizo pausible el uso de este método. 

##### Creación de funciones previas necesarias y diccionarios de mapeo para pseudonimización

Los pasos 1 y 4 de la metodología han requerido la creación previa de funciones de limpieza de carácteres invisibles e imputación de nulos

In [ ]:
# ---------- Funciones de ayuda ----------

# Función de supresión de caracteres invisibles
def suprimir_caracteres_invisibles(col):
    """
    Reemplaza caracteres invisibles por espacio (evita unir palabras), trim y colapsa whitespace.
    Devuelve campo string limpio.
    """
    cleaned = F.regexp_replace(col.cast(StringType()),
                               r'[\u0000-\u001F\u007F\u180E\u200B\u2060\uFEFF\u00A0]',
                               ' ')
    collapsed = F.regexp_replace(F.trim(cleaned), r'\s+', ' ')
    return collapsed

# Función de imputación de valores nulos en columnas String
def imputar_nulos_literal_expr(col):
    """Devuelve col cast-string y con ""/null => literal 'NULL'."""
    col_str = col.cast(StringType())
    col_trim = F.trim(col_str)
    return F.when(col_trim.isNull() | (col_trim == ""), F.lit("NULL")).otherwise(col_trim)

# En el paso metodológico 6, se menciona una condición simple para imputación de valores nulos en el clúster "age".
# Se define un umbral para imputar campos numéricos por la media (fracción máxima de nulos aceptable)
NULL_FRAC_THRESHOLD = 0.05   # 5% -> porcentaje de nulos en el umbral o inferior, se considera "bajo" y se imputarán por la media.

##### Función de limpieza para **forEachBatch**()

La función que continua se pasa al método *forEachBatch()* que transformará cada Streaming Dataframe en un Dataframe en batch

In [ ]:
# Importación de librerias necesarias
import json # Necesaria para cargar mapa de pseudónimos

In [ ]:
# El mapa con los seudónimos a utilizar en los campos sensibles se carga desde un fichero JSON preparado:
maps_input_dir = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Maps\Voters\mapa_voters.json"
with open(maps_input_dir, "r") as file:
    mapping_dicts = json.load(file)

# Dataset de votantes en Dremio
DREMIO_DATASET_VOTERS = "local_delta.Voters"  

In [ ]:
# Celda 3: función foreachBatch: limpia, transforma y escribe en Delta

def foreach_batch_voters_clean(batch_df, batch_id):
    """
    Aplica la metodología sobre batch_df (DataFrame, microbatch) y escribe el resultado en silver_voters_path (parquet).
    """
    # if batch_df.rdd.isEmpty():
    if not batch_df.head(1):
        print(f"[batch {batch_id}] vacío — nada que procesar")
        return
        
    df = batch_df
# ------------------------------------------------------------------------------------------------------------------------------------------------------------
    # Paso 1. Campos con carácteres invisibles -> normalize + trim (aplicar a city, party, ethnic_description)
    for c in ["city", "party", "ethnic_description"]:
        if c in df.columns:
            df = df.withColumn(c, suprimir_caracteres_invisibles(F.col(c)))
# ------------------------------------------------------------------------------------------------------------------------------------------------------------
    # Paso 2. Campos que requieren title -> party, city (error emergente), ethnic_description 
    for c in ["party","city","ethnic_description"]:
        if c in df.columns:
            # initcap(lower(...)) produce Title case
            df = df.withColumn(c, F.when(F.col(c).isNotNull(), F.initcap(F.lower(F.col(c)))).otherwise(None))
# ------------------------------------------------------------------------------------------------------------------------------------------------------------            
    # Paso 3. Campos en mayúscula por convención: state
    if "state" in df.columns:
        df = df.withColumn("state", F.when(F.col("state").isNotNull(), F.upper(suprimir_caracteres_invisibles(F.col("state")))).otherwise(None))
# ------------------------------------------------------------------------------------------------------------------------------------------------------------
    # Paso 4. Imputación de strings a "NULL" excepto voter_id (clave primaria) y us_congress (clave de partición)
    # Identificar todas las columnas string en el batch y aplicar imputación de nulos a excepción de los dos campos excluidos que son obligatorios 
    schema = df.schema
    string_cols = [f.name for f in schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        if c in ("voter_id", "us_congress"):
            continue
        df = df.withColumn(c, imputar_nulos_literal_expr(F.col(c)))
# ------------------------------------------------------------------------------------------------------------------------------------------------------------
    # Paso 5. Pseudonimización reversible sobre cuasi-identificadores y variables dentro de las categorías de datos especiales.
    # Se aplica mapping después de las transformaciones anteriores
    for field, mapping in mapping_dicts.items():
        # crear literal map (pares key, value)
        if not mapping:
            continue
        flat = []
        for k, v in mapping.items():
            flat.extend([F.lit(str(k)), F.lit(str(v))])
        map_col = F.create_map(*flat)  # crea un mapa literal en la expresión SQL
        
        # aplicamos la sustitución en una sola expresión: map(field) ?? original
        df = df.withColumn(
            field,
            F.coalesce(map_col[F.col(field)], F.col(field))  
        )
# ------------------------------------------------------------------------------------------------------------------------------------------------------------  
    # Paso 6. Imputación numérica por media para campos con baja fracción de nulos (campo sometido: age)
    if "age" in df.columns:
        # Se define la expresión cast solo para usarla en la agregación
        age_double = F.col("age").cast(DoubleType())
    
        # UNA sola agregación: total, non_nulls y mean
        stats = df.agg(
            F.count(F.lit(1)).alias("total"),
            F.count(age_double).alias("non_nulls"),
            F.avg(age_double).alias("mean_age")
        ).first()  # trae un solo row al driver pequeño puesto que solamente tiene 3 campos
    
        # Seguridad en extracción de valores
        total = int(stats["total"] or 0)
        non_nulls = int(stats["non_nulls"] or 0)
        mean_age = stats["mean_age"]  # puede ser None
    
        null_frac = 1.0 if total == 0 else 1.0 - (float(non_nulls) / float(total))
    
        # Si la media existe y la fracción nula es aceptable, deben imputarse los valores nulos con la media redondeada
        if mean_age is not None and null_frac <= NULL_FRAC_THRESHOLD:
            imputed_value = int(round(mean_age))
    
            # Se imputa en una sola transformación:
            df = df.withColumn(
                "age",
                F.when(F.col("age").isNull(), F.lit(imputed_value))
                 .otherwise(F.col("age").cast(IntegerType()))
            )
        else:
            # No imputar en este caso (se deberán tomar medidas más robustas en este caso): sólo casteamos cuando sea posible (los no numéricos seguirán como null)
            df = df.withColumn("age", F.col("age").cast(IntegerType()))
# ------------------------------------------------------------------------------------------------------------------------------------------------------------    
    streamingVBCuratedDF = df    
    # ---------------------------------------------------------------------------------------------
    # Controlar número de ficheros de salida
    #    - Si se requiere 1 fichero TOTAL: usar coalesce(1)  (solo para pruebas con pequeños datos)
    #    - Si se requiere paralelismo por partición: repartition("us_congress") es ideal
    # ---------------------------------------------------------------------------------------------

    # Como en el Job de Spark para ELT de carga de la capa Bronze de llamadas, puesto que los ficheros parquet son pequeños, previo a la escritura de los Streaming Dataframes curados
    # se utiliza la transformación wide repartition(1) para consolidar las particiones del RDD miembro del Dataframe en una única partición, de modo que se escriba un único fichero
    # en la capa Silver de votantes. Así, el Spark Job que se desarrolla ejecuta un mapeo 1-1 de los ficheros de la capa Bronze de votantes. 
    FORCE_SINGLE_OUTPUT_FILE = True   # Se define en True solo para pruebas pequeñas
    if FORCE_SINGLE_OUTPUT_FILE:
        streaming_voters_to_write_delta = streamingVBCuratedDF.coalesce(1)
    else:
        # repartition por la columna de partición ayuda a agrupar datos del mismo us_congress
        streaming_voters_to_write_delta = streamingVBCuratedDF.repartition(F.col("us_congress"))
    # -------------------------------------------------------
    # Escribir a Delta (append) particionado por us_congress
    # El checkpoint de streaming se gestiona en el writeStream principal,
    # pero aquí cada batch escribe de forma batch (path/silver).
    # -------------------------------------------------------    
    streaming_voters_to_write_delta.write.format("delta") \
            .mode("append") \
            .partitionBy("us_congress") \
            .save(silver_voters_path)
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    # ------- Código de actualización de metadatos------- 
    # Identificar particiones afectadas a través de la extracción de los valores distintos de la clave de partición en el Streaming Dataframe leído. 
    partitions = [row["us_congress"] for row in streaming_voters_to_write_delta.select("us_congress").distinct().collect()]

    # llamar Dremio por partición
    for p in partitions:
        # Se usan las funciones creadas en la fase de actualización de metadatos asociadaos a la tabla Delta de 
        sql = build_partition_sql(DREMIO_DATASET_VOTERS, "us_congress", p) 
        resp = post_sql(sql)
        print("Dremio response:", resp)

#### Carga de datos
En esta etapa, cada Streaming Dataframe limpiado y transformado deberá escribirse en disco al sumidero dentro de la capa Silver de votantes. 

In [ ]:
# Ruta de almacenamiento de checkpoint para evitar reprocesamiento y/o omisión de procesamiento de ficheros cuando se reanude, reinice o falle el Job de Spark que sea crea en este Notebook de Jupyter
# checkpoint_path_bronze_voters_processed = "C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center Prueba/Checkpoints/silver_voters_checkpoint" # Se almacena en la ruta de checkpoints respectiva.
checkpoint_path_bronze_voters_processed = "C:/Users/TPG/Documents/Trabajos UNIR/Semestre II/TFM/Data Call Center All/Checkpoints/silver_voters_checkpoint" # Checkpoint Voters Processed All
# Se escribe el dataframe de votantes curado
query_etl_silver_voters = (
    streamingVBReducedDF
        .writeStream
        .foreachBatch(foreach_batch_voters_clean)
        .option("checkpointLocation", checkpoint_path_bronze_voters_processed) # Para testear el funcionamiento del Job de Spark, comente esta línea de código para evitar tener que eliminar también rutas de checkpoints.
        .trigger(once=True) # Se procesan todos los archivos disponibles y se finaliza el flujo ETL (Ideal para flujos ETL sobre ficheros que arriban esporadicamente)
        .start()
)

query_etl_silver_voters .awaitTermination() # (ELIMINAR SI SE DESEA QUE EL PROCESO SE EJECUTE EN SEGUNDO PLANO)
print("Trigger.Once de flujo ETL de votantes finalizado.")

In [ ]:
import pandas as pd

In [ ]:
# Definición de paths de ficheros
path_1 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Voters\us_congress=CD-AMH\part-00000-fa615305-cd6a-44b4-aa6c-eb055ec640d0.c000.snappy.parquet"
path_2 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Voters\us_congress=CD-ARB\part-00000-a87de446-7bd0-414c-8da2-2ba4d05bc19d.c000.snappy.parquet"
path_3 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Voters\us_congress=CD-AVK\part-00000-1daaca5a-a7d8-4e32-934b-29894658a5bc.c000.snappy.parquet"
path_4 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Voters\us_congress=CD-BMH\part-00000-952de280-ce7f-4d22-ae1d-84985f81ec6f.c000.snappy.parquet"
path_5 = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Prueba\Silver\Voters\us_congress=CD-BRB\part-00000-5f416e13-8620-4f63-9550-8cce844be54c.c000.snappy.parquet"

In [ ]:
df_verificacion = pd.concat([pd.read_parquet(path_1), pd.read_parquet(path_2), pd.read_parquet(path_3), pd.read_parquet(path_4), pd.read_parquet(path_5)])
df_verificacion

In [ ]:
df_verificacion.info()

#### ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Monitoreo de flujos ETL simultáneos

En este capítulo se tiene como objetivo monitorear la ejecución de los flujos de Streaming de la arquitectura Lakehouse

In [ ]:
# Importación de librerias necesarias
import time
from datetime import datetime
from rich.table import Table
from rich.live import Live
from rich.console import Console

In [ ]:
# Para detener el monitoreo: Usar Ctrl + C para no interrumpir todo el script 
console = Console()

# Esta función crea una tabla Rich y la va poblando con la información de cada flujo Streaming 
def construir_tabla_stream():
    """
    Devuelve una tabla Rich con el estado de todos los StreamingQueries activos.
    """
    table = Table(title="Estado de Streams Activos", show_lines=True)

    # Definición de columnas
    table.add_column("Name", style="cyan", no_wrap=True)
    table.add_column("ID", style="white")
    table.add_column("Active", style="bold green")
    table.add_column("Status", style="magenta")
    table.add_column("BatchId", justify="right", style="yellow")
    table.add_column("InputRows", justify="right", style="bright_blue")
    table.add_column("ProcTime (ms)", justify="right", style="bright_yellow")
    table.add_column("LastTrigger", style="bright_white")

    # Recolectar datos de Spark
    if not spark.streams.active:
        table.add_row("—", "—", "—", "No hay consultas", "—", "—", "—", "—")
    else:
        for q in spark.streams.active:
            progress = q.lastProgress or {}
            batch_id = str(progress.get("batchId", "—"))
            input_rows = str(progress.get("numInputRows", "—"))
            proc_time = str(progress.get("durationMs", {}).get("process", "—"))
            trigger_time = progress.get("timestamp", None)

            if trigger_time:
                try:
                    trigger_time = datetime.fromisoformat(trigger_time).strftime("%Y-%m-%d %H:%M:%S")
                except Exception:
                    pass
            else:
                trigger_time = "—"

            table.add_row(
                q.name or "—",
                str(q.id) or "—",
                "✅" if q.isActive else "❌",
                q.status.get("message", "—"),
                batch_id,
                input_rows,
                proc_time,
                trigger_time
            )

    return table

In [ ]:
# Monitoreo en vivo con refresco cada 5 segundos
with Live(construir_tabla_stream(), refresh_per_second=0.5, console=console) as live:
    try:
        while True:
            time.sleep(5)  # Intervalo de actualización
            live.update(construir_tabla_stream())
    except KeyboardInterrupt:
        console.print("\n[bold red]Monitoreo detenido manualmente.[/bold red]")